# 11 RAG项目单元测试与BUG调试


## 1. 对应课程与本篇范围

上一篇完成了 RAG 项目的需求和方案整理，也核对了上传、检索、生成、引用和会话持久化的源码。本篇继续解决初版之后的问题：**怎样用测试检查组件行为，再根据失败证据定位和修复 BUG。**

对应网盘视频：

```text
22-RAG项目-单元测试和BUG调试.mp4
```

课程见 [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码 `1234`。配套讲义为 [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)，共享工程位于网盘 `项目/LangChainRAG项目/`。

讲义正文保留的是项目初始化需求，没有本节完整测试对话和报错记录。因此本篇以**已读取的后端测试、前端状态测试、模型联调脚本及对应实现**为依据整理。视频未逐段观看或转录；共享快照不能还原课堂每一次失败与修复的先后顺序。标为“补充”的用例、提示词和命令，是针对这些源码的解释与练习。

本篇保留单元测试、模型连通性定位和功能回归的主线。下一节的 Skills、Agent 迁移与 Git 提交，以及后续并发压力测试，另行整理。


## 2. 先区分三种检查，才能解释测试结果

RAG 项目同时包含普通程序逻辑、数据库、浏览器状态和外部模型服务。它们的检查范围不同。

| 检查层级 | 本项目例子 | 能说明什么 |
|---|---|---|
| 单元测试 | 密码哈希、引用格式化、分块、前端状态变化 | 给定输入或模拟依赖时，某个组件是否符合断言 |
| 集成与连通性检查 | 真实调用百炼、写入向量库、保存数据库消息 | 多个实际组件能否连接和协作 |
| 完整功能回归 | 登录、上传、等待索引、提问、查看引用、恢复历史 | 用户能否走完实际业务流程 |

例如 `format_docs_with_sources()` 的测试直接构造 `Document` 和分数，不需要真的调用嵌入模型。前端登录状态测试使用 Mock API：Mock 是用可控制的模拟依赖代替真实请求，使测试能稳定触发成功或失败分支。

真实模型调用还受账号权限、端点、网络、额度和服务状态影响，适合单独检查。不能把“模拟登录返回成功”报告成“后端真实登录成功”，也不能把模型回复了一句“你好”报告成知识库问答已经正确。

共享根目录的 `report.html` 已核对为 **Locust 压测报告**，不是 pytest 或 Vitest 的单元测试报告。本篇不引用它来证明下面的单元测试通过。


## 3. 原项目已有的测试与覆盖范围

后端 `backend/tests/` 中与本篇相关的文件如下。数量是对源码中 `test_` 方法的静态统计，**不是本次执行收集或通过的数量**。

| 文件 | 方法数 | 主要检查 |
|---|---:|---|
| `conftest.py` | — | 导入路径、事件循环夹具 |
| `test_config.py` | 7 | 配置值、API Key 字符串、扩展名、CORS、数据库和 Chroma 配置 |
| `test_security.py` | 12 | bcrypt 哈希与验证、JWT 创建和解码 |
| `test_splitter.py` | 8 | 短文本、长文本、空文本、多文档、中文和换行等输入 |
| `test_rag_chain.py` | 8 | 引用格式化六种情况、提示词关键规则两种检查 |
| **四个测试文件合计** | **35** | 组件逻辑；不等于 35 条完整业务流程 |

前端已读取的两组状态测试：

| 文件 | `it()` 用例数 | 主要检查 |
|---|---:|---|
| `frontend/src/store/authStore.test.ts` | 9 | 登录、注册、登出和缓存初始化 |
| `frontend/src/store/chatStore.test.ts` | 9 | 会话列表、创建、切换、删除及部分流式状态 |

另外还有 `backend/test_api.py`、`backend/test_dashscope.py` 两个真实模型联调脚本。它们虽然也以 `test_` 命名，但主要在模块顶层发起请求并打印结果，不是与上面四个文件同类的隔离单元测试。

测试文件名、用例名称和实际断言要一起看。例如 `test_rag_chain.py` 文件说明提到了检索器，但正文主要验证引用格式化和提示词，没有执行真实检索、流式生成和模型回答质量验证。


## 4. 运行后端测试：明确解释器和收集范围

沿用 10 的项目环境，在 `backend/` 中使用本项目 `venv` 的 Python。原 `requirements.txt` 没有列出 pytest，下面安装测试工具的命令是补充步骤，不是原清单的内容；实际项目应把确认兼容的测试工具版本记录到开发依赖中。

```powershell
cd backend
.\venv\Scripts\python.exe -m pip install pytest
```

先明确四个单元测试文件，再收集用例。这组命令在同一个 PowerShell 会话中执行：

```powershell
$unitFiles = @(
    'tests/test_config.py'
    'tests/test_security.py'
    'tests/test_splitter.py'
    'tests/test_rag_chain.py'
)
.\venv\Scripts\python.exe -m pytest @unitFiles --collect-only -q
.\venv\Scripts\python.exe -m pytest @unitFiles -v
```

`--collect-only` 查看将要运行的测试，仍然会导入模块。显式选择这四个文件，能把本轮范围限定到已核对的单元测试，避免宽泛收集时导入根目录联调脚本并触发其顶层模型请求。

失败后先缩小到单个模块或用例：

```powershell
.\venv\Scripts\python.exe -m pytest tests/test_security.py -v -x --tb=short
.\venv\Scripts\python.exe -m pytest 'tests/test_security.py::TestJWTToken::test_NOne_token_返回None' -v
```

`-x` 在首个失败时停止，`--tb=short` 显示较短的异常栈，`文件::类::方法` 精确选择用例。原方法名里的 `NOne` 就是源码写法，不要改成另一个拼写后照抄节点路径。

需要保存机器可读结果时，可使用 pytest 自带的 JUnit XML 输出。在上述 `$unitFiles` 已定义的终端中执行：

```powershell
New-Item -ItemType Directory -Path reports -Force | Out-Null
.\venv\Scripts\python.exe -m pytest @unitFiles -v --junitxml=reports/unit-tests.xml
```

以上都是学习操作，没有在本次整理中执行。缺少模块属于导入或环境问题；`AssertionError` 属于断言不满足；没有收集到用例也不表示测试通过。应先读取真实命令、退出状态和错误位置，再决定改哪里。


## 5. pytest 的用例、断言与夹具

原测试使用 `Test...` 类组织场景，方法名以 `test_` 开头，使用 `assert` 表达预期。例如“正常密码哈希后验证通过”，应同时检查输出不是明文，以及正确密码能通过验证。

`tests/conftest.py` 原文如下：

```python
"""pytest 配置文件 — 测试夹具和全局配置"""

import pytest
import os
import sys
import asyncio

# 确保项目根目录在 sys.path 中
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))


@pytest.fixture(scope="session")
def event_loop():
    """创建事件循环用于异步测试"""
    loop = asyncio.new_event_loop()
    yield loop
    loop.close()
```

`sys.path` 处理让测试能导入 `app`；这里加入的是相对 `tests/` 的上一级，也就是后端目录。Fixture（夹具）负责准备和清理测试资源，`yield` 前创建资源，测试使用后执行清理。

该 `event_loop` 夹具作用域为整个测试会话。已读取的 35 个方法主要是同步组件测试，没有因为存在这个夹具就自动变成异步测试。后续增加异步用例时，还要根据实际 pytest 与异步插件版本确定执行方式，不能只复制夹具就认为协程已被等待。

一个有用的测试通常包含：构造有区别的输入、调用待测组件、断言与需求相关的结果。只有“函数没有报错”或“返回值非空”，往往不足以证明复杂功能。


## 6. 密码与 JWT：正常输入和异常输入都要检查

`test_security.py` 将密码哈希与 JWT 分成两组，各有 6 个方法。

| 分组 | 原用例覆盖 |
|---|---|
| 密码 | 正确密码、错误密码、空密码、同密码不同盐、50 字符密码、特殊字符与中文 |
| JWT | 创建后解码、包含 `exp`、伪造 token、空字符串、`None`、普通用户角色 |

下面摘录密码测试类中的方法，保留原缩进：

```python
    def test_相同密码两次哈希_结果不同(self):
        """场景：相同密码两次哈希 → 生成不同的哈希值（盐不同）"""
        password = "mypassword"
        hash1 = hash_password(password)
        hash2 = hash_password(password)

        assert hash1 != hash2  # 不同的盐
        assert verify_password(password, hash1) is True
        assert verify_password(password, hash2) is True
```

相同密码两次哈希不同，但都能验证通过，检查的是盐参与哈希后的行为。空密码能被底层哈希函数处理，只代表这个函数的输入行为，不代表注册接口应允许空密码；注册请求的数据校验属于另一个检查位置。

原 JWT 测试中有一条异常输入用例：

```python
    def test_NOne_token_返回None(self):
        """场景：None → decode 返回 None"""
        payload = decode_access_token(None)
        assert payload is None
```

对应 `app/core/security.py` 当前实现：

```python
def decode_access_token(token: Optional[str]) -> Optional[Dict[str, Any]]:
    """解码 JWT 令牌，返回 payload 或 None（无效/过期时）"""
    if not token:
        return None
    try:
        payload = jwt.decode(
            token,
            settings.JWT_SECRET,
            algorithms=[settings.JWT_ALGORITHM],
        )
        return payload
    except (JWTError, AttributeError):
        return None
```

这份快照先处理空值，再捕获指定的解码异常。它提供了“异常输入 → 明确返回值”的回归目标；没有课堂旧版本或失败栈，就不能编造某次报错的完整修复历史。

原“包含过期时间”用例只检查 `exp` 存在，没有检查已经过期的 token 是否被拒绝。可以在同一测试模块中补充以下回归用例，使用真实 JWT 签名与解码函数，不访问外部模型：

```python
def test_已过期token_解码返回None():
    from datetime import timedelta

    token = create_access_token(
        {"sub": "1", "role": "user"},
        expires_delta=timedelta(minutes=-1),
    )
    assert decode_access_token(token) is None
```

这是补充示例，不属于已读取的 12 个原方法。本次没有执行它。JWT 工具函数通过，也不等于 HTTP 接口已经正确检查用户启用状态、管理员角色和会话归属。


## 7. 配置测试：区分默认声明、环境加载和连通性

`test_config.py` 直接导入全局 `settings`，检查模型、分块、阈值、管理员、数据库和跨源配置。API Key 用例原文为：

```python
    def test_DashScope_API_Key_已配置(self):
        """场景：百炼 API Key 不为空"""
        assert settings.DASHSCOPE_API_KEY != ""
        assert len(settings.DASHSCOPE_API_KEY) > 10
        assert settings.DASHSCOPE_API_KEY.startswith("sk-")
```

它检查字符串是否符合几个形式条件，没有向百炼发送请求，所以不能证明密钥可用、模型已开通或端点匹配。

另一个边界是测试环境：`Settings` 会读取 `.env` 和进程环境变量。原测试方法名写“默认值”，实际断言的是已经加载后的全局配置。如果开发者主动把 `CHUNK_SIZE` 改成 1200，断言固定为 800 的用例会失败；这不一定表示配置加载器有 BUG。

**补充：**将两种意图区分成独立用例。默认声明可以读取 Pydantic 字段定义；环境加载则用 pytest 的 `monkeypatch` 显式提供输入，并创建新实例：

```python
from app.config import Settings


def test_分块大小_默认声明():
    assert Settings.model_fields["CHUNK_SIZE"].default == 800


def test_环境变量_覆盖分块大小(monkeypatch):
    monkeypatch.setenv("CHUNK_SIZE", "1200")
    loaded = Settings(_env_file=None)
    assert loaded.CHUNK_SIZE == 1200
```

`_env_file=None` 在这次实例化中关闭 `.env` 文件读取，环境变量仍生效；`monkeypatch` 在用例结束后恢复修改。这里没有修改已经创建的全局 `settings`，也没有假定变更环境变量会让旧实例自动重新加载。

配置断言失败时，先核对测试是在验证默认值还是某个部署环境，再判断预期应保持还是调整。不要用一个环境的配置结果概括另一个环境。


## 8. 分块测试：断言要对应切分行为

`test_splitter.py` 每个方法执行前，通过 `setup_method()` 调用 `get_text_splitter()` 创建分割器。原项目默认长度为 800、重叠为 100，长度函数是 `len`。

原用例覆盖短文本、长文本、中文句号、空文本、多文档、换行、中英混合和重叠。多文档用例同时检查来源元数据：

```python
    def test_多文档_分别分割(self):
        """场景：多个文档 → 每个文档都被分割"""
        text1 = "文档一内容。" * 100
        text2 = "文档二内容。" * 100
        docs = [
            Document(page_content=text1, metadata={"source": "doc1"}),
            Document(page_content=text2, metadata={"source": "doc2"}),
        ]
        chunks = self.splitter.split_documents(docs)

        assert len(chunks) > 0
        # 验证元数据被保留
        sources = {c.metadata.get("source") for c in chunks}
        assert "doc1" in sources
        assert "doc2" in sources
```

这条用例能检查两个来源仍出现在结果中；由于这两段示例各自较短，不能据此证明每份文档都被拆成了多个块。元数据保留很重要，因为后面的引用和按文档管理需要它。

原重叠用例的关键断言如下，摘自方法内部：

```python
        if len(chunks) >= 2:
            # 相邻块应该有内容重叠
            last_chunk = chunks[0].page_content
            next_chunk = chunks[1].page_content
            # overlap 导致尾部内容出现在下一个块中
            # 简单地验证两个块不是完全独立的
            assert len(last_chunk) > 0
            assert len(next_chunk) > 0
```

它实际只检查两个块非空；若根本没产生两个块，条件体还会被跳过。类似地，中文句号用例检查多块和非空，没有断言切分位置一定处于句子边界。应按断言解释覆盖范围，不能只照用例名称报告。

**补充：**针对“无分隔符时重叠 100 字符”，构造没有重复字符的输入，固定本用例的参数，再直接比较相邻块边界。与重复短词相比，这种输入更容易发现重叠位置错误。

```python
from app.config import settings
from app.rag.splitter import get_text_splitter
from langchain_core.documents import Document


def test_无分隔符文本_相邻块重叠100字符(monkeypatch):
    monkeypatch.setattr(settings, "CHUNK_SIZE", 800)
    monkeypatch.setattr(settings, "CHUNK_OVERLAP", 100)
    text = "".join(chr(0x4E00 + i) for i in range(2000))
    chunks = get_text_splitter().split_documents([Document(page_content=text)])

    assert len(chunks) >= 2
    assert len(chunks[0].page_content) == 800
    assert chunks[0].page_content[-100:] == chunks[1].page_content[:100]
```

这个补充用例只针对明确构造的输入，不要求所有按自然段切分的文档都恰好重叠 100 字符；递归分割器的分隔符和合并过程也会影响边界。


## 9. RAG 引用测试：检查结构与边界，不调用大模型

`test_rag_chain.py` 的 `TestFormatDocs` 直接构造检索结果，再调用 `format_docs_with_sources()`。

| 原用例 | 实际预期 |
|---|---|
| 单个文档 | 上下文包含来源编号和正文，来源字段与元数据一致 |
| 多个文档 | 按输入顺序编号为来源 1、2、3 |
| 空列表 | 返回空上下文和空来源列表 |
| 长片段 | 来源预览长度为 300 |
| 浮点分数 | `0.12345678` 四舍五入为 `0.1235` |
| 缺失元数据 | 文件名为“未知文件”，块序号为 0，品类为 `None` |

以下是原测试类中的截断用例：

```python
    def test_来源文本预览_截断正确(self):
        """场景：chunk 文本超过 300 字 → 预览截断到 300 字"""
        long_text = "A" * 500
        doc = Document(
            page_content=long_text,
            metadata={"filename": "test.md", "chunk_index": 0},
        )

        _, sources = format_docs_with_sources([(doc, 0.9)])

        assert len(sources[0]["chunk_text"]) == 300  # 预览截断到 300
```

实现中给模型的上下文保留完整 `page_content`，来源列表只保留前 300 字符。原用例检查了预览长度，可以进一步增加“上下文仍然完整”的回归断言：

```python
def test_引用预览截断_不截断模型上下文():
    text = "A" * 300 + "这里是片段尾部的关键参数"
    doc = Document(page_content=text, metadata={"filename": "product.md"})

    context, sources = format_docs_with_sources([(doc, 0.9)])

    assert text in context
    assert sources[0]["chunk_text"] == text[:300]
    assert "关键参数" not in sources[0]["chunk_text"]
```

这是同一测试模块中可添加的补充函数，沿用原文件的 `Document` 和待测函数导入。它解释两种文本的不同用途，不是在修改产品的预览长度。

`TestSystemPrompt` 的两条测试检查提示词包含“参考文档”“无法找到相关信息”“引用来源”“中文回答”“表格形式”和 `{context}` 等文字。这能发现模板被误删或遗漏规则，不能证明模型一定拒绝编造，也不能证明回答中的每个来源编号都正确。

多文档测试同样不验证检索排序算法：函数接收到什么顺序，就按什么顺序编号；真正的相关性检索需要另设输入和预期。


## 10. 前端测试环境：Vitest、jsdom 与测试初始化

`frontend/package.json` 提供下列原脚本：

```json
{
  "test": "vitest run",
  "test:watch": "vitest",
  "test:coverage": "vitest run --coverage"
}
```

以下为已取得的 `frontend/vitest.config.ts` 原文：

```typescript
import { defineConfig } from 'vitest/config'
import react from '@vitejs/plugin-react'
import { resolve } from 'path'

export default defineConfig({
  plugins: [react()],
  test: {
    environment: 'jsdom',
    include: ['src/**/*.test.{ts,tsx}'],
    globals: true,
    testTimeout: 10000,
  },
  resolve: {
    alias: {
      '@': resolve(__dirname, 'src'),
    },
  },
})
```

`jsdom` 提供测试用的 DOM 环境；`include` 收集 `src/` 下 `.test.ts` 和 `.test.tsx`；单个测试超时设为 10 秒；`@` 指向 `src/`。

工程还包含 `src/test-setup.ts`，其中导入 `@testing-library/jest-dom` 并创建内存版 `localStorage`。但上述配置没有 `setupFiles`，已读取的两份状态测试也没有导入该文件。因此“文件存在”不能证明初始化已自动生效；这两组测试可以使用 jsdom 自带的 `localStorage`，不应据此断言它们必然无法运行。

若新增测试明确依赖这个初始化文件，可以在 `test` 配置中补充下面一项，并重新验证。这属于补充配置：

```typescript
setupFiles: ['./src/test-setup.ts'],
```

从项目根目录另开终端，进入前端目录：

```powershell
cd frontend
npm install
npm test
npm run build
```

`npm test` 执行一次测试，`npm run test:watch` 适合边修改边观察；`npm run build` 使用 `tsc -b && vite build`，检查构建是另一个步骤。测试通过不能代替 TypeScript 构建通过。

原文件有覆盖率脚本，但已读取的依赖清单未列出覆盖率 provider 包。需要生成覆盖率时，应先确认安装了与当前 Vitest 版本兼容的 provider，再运行 `npm run test:coverage`；脚本名称存在不等于覆盖率报告已经生成。


## 11. 登录状态测试：模拟 API，再检查状态和缓存

`authStore.test.ts` 使用 `vi.mock("../api/auth", ...)` 将登录、注册和当前用户查询等函数替换为 Mock。每条测试之前清空 `localStorage`、重置 Zustand 状态，并调用 `vi.clearAllMocks()` 清理调用记录。

管理员登录成功用例如下，摘自原测试：

```typescript
    it("登录成功：设置 user、token、isAuthenticated、isAdmin", async () => {
      const mockUser = { id: 1, username: "admin", role: "admin", created_at: "2024-01-01" };
      const mockToken = "jwt-token-abc";
      vi.mocked(authApi.login).mockResolvedValue({
        access_token: mockToken,
        token_type: "bearer",
        user: mockUser,
      });

      await useAuthStore.getState().login("admin", "123456");

      const state = useAuthStore.getState();
      expect(state.token).toBe(mockToken);
      expect(state.user).toEqual(mockUser);
      expect(state.isAuthenticated).toBe(true);
      expect(state.isAdmin).toBe(true);
      expect(state.loading).toBe(false);
      expect(localStorage.getItem("access_token")).toBe(mockToken);
    });
```

`mockResolvedValue` 安排一个成功响应；调用真实的 Store 登录方法后，断言令牌、用户、认证状态、管理员标记和缓存。普通用户用例另检查 `isAdmin=false`，失败用例则安排 Promise 拒绝并检查错误向上传递、`loading` 恢复。

其余用例检查注册成功和失败、登出清理、无 token 时初始化、有缓存时恢复、普通用户缓存的管理员标记。它们验证的是前端状态逻辑，不验证后端密码、JWT 签名或真实权限。

`initialize()` 还有一个异步边界：它先读取缓存，再调用 `fetchCurrentUser()` 刷新用户。原缓存恢复用例同步检查初始状态，没有等待这次刷新，也没有给 `getMe()` 安排完整返回值。由此不能证明“缓存恢复后服务端校验也成功”。

增加这类用例时，应分别模拟 `getMe()` 成功与拒绝，并等待刷新完成后检查最终状态。`vi.clearAllMocks()` 清理调用记录，不等于重置所有 Mock 实现和返回队列；需要按每条用例显式设置依赖行为，避免测试之间共享未预期的状态。


## 12. 聊天状态测试：会话操作与流式回调分开验证

`chatStore.test.ts` 模拟了会话 API 与 `sendChatMessage()`，每条测试前重置消息、活跃会话、流式内容、引用、加载状态和 `abortController`。

原 9 条用例检查：会话列表成功和失败、创建会话返回 ID、切换会话加载历史、删除活跃会话清理消息、没有活跃会话时返回、流式输出中防止重复发送、空闲时停止、清空消息与流式内容。

下面是原“正在输出时不重复发送”用例：

```typescript
    it("正在流式输出中：不重复发送", async () => {
      useChatStore.setState({ activeConversationId: 1, isStreaming: true });
      await useChatStore.getState().sendMessage("another message");
      // 状态不变
      expect(useChatStore.getState().messages).toHaveLength(0);
    });
```

它检查消息数组没有增加，没有直接断言发送 API 未被调用。若要把用例意图表达得更完整，可在同一模块增加 API 调用断言，或对现有用例补充：

```typescript
import * as chatApi from "../api/chat";

expect(vi.mocked(chatApi.sendChatMessage)).not.toHaveBeenCalled();
```

这里是两个应放在各自位置的补充行：导入放在模块导入区，`expect` 放在对应用例调用之后。不能把这两行当作独立完整测试执行。

当前测试没有用 Mock 主动触发 `onToken`、`onSources`、`onDone` 和 `onError`，因此还不能证明“增量拼接 → 引用接收 → 完成后落入消息列表”的整条状态转换。为这些回调安排可控制的触发顺序，才能检查引用是否挂到正确的回答、完成后是否退出流式状态。

此外，Mock 替换了网络层，状态测试通过仍不能证明 SSE 解析正确或后端消息已保存；这些要通过真实接口与刷新历史的回归确认。


## 13. BUG 定位练习：第一段文字到来前停止生成

这是根据已读取的 `chatStore.ts` 推导的补充调试案例，不是已取得课堂报错记录。

当前 `sendMessage()` 先设置 `isStreaming=true`，再保存发送函数返回的 `AbortController`。`stopStreaming()` 会调用 `abort()`，但清理 `isStreaming` 等字段的 `set()` 位于 `if (state.streamingContent)` 内。

如果用户在第一段文字到来前停止，`streamingContent` 还是空字符串，取消请求后可能跳过状态清理。网络层是否另行回调也会影响实际表现，因此需要一个隔离的用例验证 Store 自身行为。

可在原 `chatStore.test.ts` 中增加下列测试：导入区补上 `chatApi`，测试放在已有 Mock 和 `beforeEach` 生效的外层 `describe` 内。

```typescript
import * as chatApi from "../api/chat";

it("首个 token 前停止：取消请求并清理流式状态", async () => {
  const controller = new AbortController();
  vi.mocked(chatApi.sendChatMessage).mockReturnValue(controller);
  useChatStore.setState({ activeConversationId: 1 });

  await useChatStore.getState().sendMessage("测试问题");
  expect(useChatStore.getState().isStreaming).toBe(true);
  expect(useChatStore.getState().streamingContent).toBe("");

  useChatStore.getState().stopStreaming();

  expect(controller.signal.aborted).toBe(true);
  expect(useChatStore.getState().isStreaming).toBe(false);
  expect(useChatStore.getState().abortController).toBeNull();
});
```

本用例没有触发任何网络回调，明确验证“停止操作自身是否清理状态”。按当前函数体推演，后两个断言会暴露状态清理只在有文本时执行的问题；本次没有运行测试，实际失败输出仍需执行后取得。

修复方向是把“有部分文字时生成临时助手消息”和“每次停止都清理流式状态”分别处理。随后至少回归两条路径：尚无文字就停止、已有部分文字时停止。还要检查取消请求后的回调不会重复追加消息。

这体现了 BUG 调试的一条顺序：把具体操作时机转成测试输入，先证明失败落在哪个分支，再修改对应行为，最后用原失败用例和相关用例确认结果。


## 14. 模型连通性调试：分清 SDK、框架和端点

`backend/test_dashscope.py` 对比四条调用路径：

| 原脚本检查 | 目的 |
|---|---|
| DashScope SDK 的 `Generation.call()` | 核对原生 SDK 调用 |
| LangChain-Community 的 `ChatTongyi` | 核对另一种框架包装路径 |
| `ChatOpenAI` 连接公共兼容端点 | 核对 OpenAI 兼容协议路径 |
| `ChatOpenAI` 连接课程 MaaS 业务端点 | 核对账号与对应业务端点是否匹配 |

`test_api.py` 进一步对比原始 OpenAI SDK 的非流式与流式调用、LangChain `ChatOpenAI` 调用、消息形式及 `OpenAIEmbeddings`。

这些脚本保留了尝试不同调用方式的思路，但含顶层请求、写死的演示连接配置，且其中一个脚本打印密钥前缀。笔记不复制原密钥和业务端点，用项目配置表示待测连接，也不把尝试过的 SDK 全部当作最终选型。

**补充最小检查：**在 `backend/` 中保存为单独的手动联调脚本，使用自己的项目配置，真实调用模型时再执行。不要把它混入日常隔离单元测试。

```python
from app.config import settings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


def main():
    print("模型:", settings.LLM_MODEL)
    print("端点:", settings.DASHSCOPE_API_BASE)
    print("已配置 API Key:", bool(settings.DASHSCOPE_API_KEY))

    llm = ChatOpenAI(
        model=settings.LLM_MODEL,
        api_key=settings.DASHSCOPE_API_KEY,
        base_url=settings.DASHSCOPE_API_BASE,
        temperature=settings.LLM_TEMPERATURE,
    )
    response = llm.invoke([HumanMessage(content="回复两个字：你好")])
    assert isinstance(response.content, str) and response.content.strip()
    print(response.content)


if __name__ == "__main__":
    main()
```

最小调用成功之后，再逐步恢复流式、系统提示词、检索上下文和历史消息。这样每一步出现差异时，都能缩小问题范围，而不是同时改 SDK、模型名、端点和应用代码。

对话与嵌入要分别确认。共享项目最终 `rag/embeddings.py` 使用 DashScope `TextEmbedding.call()`，因此联调脚本里 `OpenAIEmbeddings` 的结果不能直接代表当前应用的嵌入路径。

原脚本多数异常被捕获后仅打印 `FAIL` 或 `Exception`，末尾仍会打印 `DONE`。看到脚本结束不等于全部成功，必须检查每条调用的结果；单纯读取进程是否正常退出，也可能遗漏被捕获的错误。


## 15. 会话与流式保存：按当前源码建立回归目标

共享项目 `api/chat.py` 已包含两处与历史和流式响应有关的处理。这里解释当前代码，不把它写成缺少证据的课堂修复前后对照。

### 先读取历史，再保存当前问题

当前接口在检查会话归属后执行：

```python
    # 2. Load chat history (must load BEFORE saving user message so it doesn't include itself)
    chat_history = await _load_chat_history(db, conv_id)

    # 3. Save user message + auto-title
    user_msg = Message(conversation_id=conv_id, role="user", content=body.content)
    db.add(user_msg)
```

模型消息构造函数随后会再追加当前问题。若把当前问题先存入、又从历史中读出，就可能重复发送。当前顺序是在代码中明确避免这种情况。

**补充单元回归：**可在 RAG 测试模块增加对消息构造函数的检查，无需请求模型：

```python
from app.rag.chain import _build_messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


def test_消息构造_历史顺序和当前问题():
    messages = _build_messages(
        "商品参考片段",
        "当前问题",
        [("human", "历史问题"), ("ai", "历史回答")],
    )

    assert isinstance(messages[0], SystemMessage)
    assert "商品参考片段" in messages[0].content
    assert isinstance(messages[1], HumanMessage)
    assert isinstance(messages[2], AIMessage)
    assert [m.content for m in messages[1:]] == ["历史问题", "历史回答", "当前问题"]
```

这条测试验证构造函数，不验证数据库读取和保存顺序。后者需要通过聊天接口与实际消息记录再做集成检查。

### 流式完成后，用独立会话保存回答

当前 `event_stream()` 收到 `done` 后，通过 `async_session_factory()` 创建独立数据库会话，写入助手回答和来源 JSON，显式 `commit()`，刷新取得消息 ID，再发出 HTTP `done` 事件。

这个处理避免流式生成器依赖外层请求数据库会话的生命周期。复查时需要观察两件事：浏览器是否收到完成事件，以及重新进入会话后是否能读到保存的回答和引用。页面已有文字只能说明流式内容到达，不能单独证明持久化成功。

回归场景应包含正常完成、生成中途失败、刷新后恢复历史，以及当前问题不重复进入模型输入。用户主动中止时的前端临时消息与后端保存行为也需要分别确认，不能用一侧的状态代替另一侧的数据结果。


## 16. 把报错交给 AI：附上复现条件和预期行为

单元测试失败后，先保留准确用例节点、异常类型、断言两边的值、相关函数和依赖版本。浏览器故障还需要操作顺序、接口状态、SSE 事件或控制台信息；数据库故障则需要对应记录和事务日志。

可以把本项目的调试过程组织为：

1. **复现。**固定输入、操作时机和运行命令，确认问题可以重复出现。
2. **分类。**区分环境导入失败、配置预期不匹配、程序断言失败、外部服务错误或前后端协议不一致。
3. **定位。**沿调用链缩小到具体函数和分支，先读取异常栈指向的代码。
4. **形成回归用例。**让用例表达需求或约定，记录修改前的真实失败。
5. **修复。**修改与原因对应的实现或不正确的测试预期，并说明理由。
6. **验证。**先重跑失败用例，再运行相关模块；涉及接口或页面行为时，补做相应功能回归。

下面是针对“首个 token 前停止”的补充提示词，不是课堂对话原文：

```text
问题：发送问题后，在第一个 token 到来前点击停止，界面仍保持生成中。
预期：请求取消后退出流式状态，可以再次发送；没有文字时不追加空助手消息。
复现条件：当前会话有效，sendChatMessage 已返回 AbortController，尚未触发 onToken。
请先检查 chatStore.ts 的 sendMessage 和 stopStreaming，以及现有 chatStore.test.ts。
增加能够固定上述时机的回归用例，给出修改前的实际失败信息。
解释原因后修复对应分支，再运行该用例、相关状态测试和前端构建。
报告修改文件、验证命令与结果；没有执行的检查明确标为待验证。
```

一次清楚的修复报告应把“用户遇到的行为”“原因”“改了哪个分支”“哪条用例验证了它”连起来。暂时不能复现时先记录缺少的条件，不把假设写成已确认原因。

如果失败来自测试把已覆盖的环境配置误当成默认值，也应修正测试的意图与输入，而不是把正确的环境加载行为改坏。测试和实现都要接受需求与证据的核对。


## 17. 单元测试之后，还要回到 RAG 主流程

本项目已有测试提供了组件级基础，但以下能力仍需额外证据：

| 能力 | 为什么现有断言不足 | 后续功能回归 |
|---|---|---|
| 真实注册登录与权限 | 前端使用 Mock，安全工具测试不经过路由 | 注册普通用户、登录并直接请求管理员接口 |
| 多用户会话隔离 | Store 测试不检查服务端归属 | 用另一普通用户读取或向已有会话 ID 发消息 |
| 文档入库 | 分割器测试不调用加载、嵌入和后台事务 | 上传课程 Markdown，等待 `indexed` 并核对分块 |
| 检索正确性 | 引用格式化使用人工构造的分数和文档 | 真实检索已知商品问题，核对命中的片段 |
| 回答依据资料 | 提示词测试只检查关键词 | 对照商品参数与引用，提问资料没有的信息 |
| SSE 与历史保存 | 状态测试没有真实 HTTP 流或数据库 | 核对完成事件，再刷新恢复正文和引用 |
| 停止生成与错误恢复 | 原聊天测试主要检查部分状态分支 | 在首段文字前、生成中和异常后分别操作 |

沿用 10 的 `data/products/electronics.md`，上传并索引后询问“索尼 WH-1000XM5 开启降噪后续航多久”，对照课程资料中的“约30小时”，再提问未提供的即时库存。测试的是回答是否依据这份教学资料，不是核实真实商品规格。

完成一轮修改后，报告实际测试范围、通过与失败数量、跳过项、构建状态和未覆盖部分。35 个后端方法和两组前端状态用例的源码数量，都不能直接写成“整个 RAG 系统全部通过”。并发与性能仍留给后续压测章节。


## 18. 来源与本次整理状态

课程来源：

- [VibeCoding实战网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)：第 22 节“单元测试和BUG调试”，沿用已核对的章节顺序。
- [Part3 LangChain RAG项目实战](https://www.yuque.com/xxcls/vibecoding/mtwqfc61tifo9cgi)：核对讲义仍以初始化需求为主，不据此编造测试对话和历史失败输出。

共享 `LangChainRAG项目/` 中使用的资料：

| 文件 | 整理依据 |
|---|---|
| `backend/tests/conftest.py` | 导入路径和事件循环夹具原文 |
| `backend/tests/test_config.py` | 7 个配置测试 |
| `backend/tests/test_security.py` | 12 个密码与 JWT 测试 |
| `backend/tests/test_splitter.py` | 8 个分块测试及断言边界 |
| `backend/tests/test_rag_chain.py` | 8 个引用与提示词测试 |
| `backend/app/core/security.py` | 空 token、JWT 解码和异常处理 |
| `backend/test_api.py`、`test_dashscope.py` | 模型与 SDK 的手动联调思路；原密钥未复制 |
| `frontend/vitest.config.ts`、`src/test-setup.ts` | 前端测试环境及初始化加载边界 |
| `frontend/src/store/authStore.test.ts`、`authStore.ts` | 登录、缓存和异步刷新行为 |
| `frontend/src/store/chatStore.test.ts`、`chatStore.ts` | 会话状态、发送与停止生成行为 |
| 前篇已读取的配置、分块器、RAG 链、聊天接口和 `package.json` | 对照待测实现、流式协议和运行命令 |
| 根目录 `report.html` | 仅核对为 Locust 报告，不作为本节单元测试结果 |

测试方法及片段来自原文件；新增回归用例、测试命令、最小模型检查和调试提示词均已标明为补充。前端状态测试文件通过单文件下载取得原文，未运行下载的工程代码。

本次只新增 11 Notebook，没有修改 01—10、课程项目源码或当前工具配置；没有安装依赖、运行课程测试、发起模型请求、修复实际工程或执行 Git 提交。格式和示例语法检查不等于项目测试通过。资料核对日期：`2026-10-10`。
